# recursive_opt × Trace-Bench — Phase 1→7 Campaign Notebook
One notebook = the whole experiment trajectory: each phase has **(a)** a gate, **(b)** the spec(s),
**(c)** a guarded live run cell, **(d)** analysis (mean±std, paired Δ), **(e)** a **decision**
(ADOPT / REJECT / PARK), and **(f)** **capitalization** — what is recorded into shared campaign
memory and carried forward *even when a PR is not merged*.

**Standing decision rule:** ADOPT iff paired same-seed Δ > 1 pooled std on ≥ 2 families at equal
budget; REJECT iff Δ < 0; otherwise PARK.

This notebook is intentionally **live-only**: it fails fast unless `OPENAI_API_KEY` is present,
`gpt-5.4-nano` passes preflight, and a real Trace-Bench adapter is registered. Saved `phase*.json`
files are used only for the final decision board, never as a fallback for failed live cells. Phase
results persist in `./campaign/` and priors/tools/skills in `./mem_campaign`.

In [1]:
import os, sys, json, time, statistics, pathlib

ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "opto").exists() and (ROOT.parent / "opto").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from opto.features.recursive_opt import (run_spec, validate_spec, compile_level,
    agentic_optimizer_factory, MemoryLite, best_config_from, register_config_values)
from opto.features.recursive_opt import tracebench as TB
from opto.features.recursive_opt.inspect_utils import repeat_scores, fmt_mean_std
from opto.features.recursive_opt.runmode import mode_banner, preflight_model

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("This notebook is live-only: set OPENAI_API_KEY before executing it.")
os.environ["RECURSIVE_OPT_MODEL"] = "gpt-5.4-nano"
os.environ["TRACE_LITELLM_MODEL"] = "gpt-5.4-nano"
os.environ.setdefault("RECURSIVE_OPT_NUM_CANDIDATES", "2")
preflight_model("gpt-5.4-nano")

TRACEBENCH = {
    "max_examples": 1,
    "inner_steps": 1,
    "inner_candidates": 1,
    "timeout_seconds": 5,
    "allowed_inner_trainers": ["MinibatchAlgorithm", "PrioritySearch"],
    "eval_kwargs": {"n_train": 1, "n_val": 0},
}
TB.configure_tracebench_adapter(TRACEBENCH, require=True)
if not TB.using_real_tasks():
    raise RuntimeError("A real Trace-Bench adapter is required; synthetic stubs are not allowed.")

CAMPAIGN = pathlib.Path("./campaign"); CAMPAIGN.mkdir(exist_ok=True)
MEM_ROOT = "./mem_campaign"
RUN = True
print(mode_banner(True))

FAMILIES = {
  "optimization_control": ["llm4ad:online_bin_packing_local", "llm4ad:optimization_admissible_set"],
  "reasoning_control": ["internal:multiobjective_gsm8k", "internal:multi_param"],
}
BUDGET = {
    "optimizer_llm_calls": 12,
    "eval_llm_calls": 24,
    "candidates": 12,
    "wall_time_s": 300,
    "on_exceed": "return_best",
}
SCORING  = {"mode": "relative_delta", "clip": [-1.0, 1.0], "report_raw": True}  # cross-scale safe
PROMOTION= {"enabled": True, "min_support": 2, "min_score": 0.05}  # score-gated: no junk priors
LIVE_SEEDS = (0,)  # bounded live campaign; use (0,1,2) for publication-quality variance.
MEASURED_TRAINERS = ["MinibatchAlgorithm", "PrioritySearch"]  # adapter-compatible trainer arms
COMPATIBILITY_ONLY_TRAINERS = ["POLCA", "ParetobasedPS"]  # valid labels, not measured under this smoke allowlist

def save_phase(name, payload): json.dump(payload, open(CAMPAIGN/f"{name}.json","w"), indent=1)
def load_phase(name):
    p = CAMPAIGN/f"{name}.json"
    return json.load(open(p)) if p.exists() else None

def run_variants(make_spec, variants, level_id, seeds=LIVE_SEEDS):
    '''Paired same-seed live runs: one spec per variant; returns {variant: stats}.'''
    out = {}
    for v in variants:
        spec = make_spec(v)
        validate_spec(spec)
        def one(seed, _v=v, _spec=spec):
            s = json.loads(json.dumps(_spec)); s["memory_root"] = MEM_ROOT
            s.setdefault("tracebench", TRACEBENCH)
            s.setdefault("budget", BUDGET)
            return run_spec(s)["results"][level_id(_v)]["score"]
        out[str(v)] = repeat_scores(one, seeds=seeds)
        print(fmt_mean_std(out[str(v)], str(v)))
    return out

def decide(stats, control_key):
    '''ADOPT / REJECT / PARK vs a control variant, per the standing rule.'''
    if not stats or control_key not in stats: return "PARK (no data)"
    c = stats[control_key]; verdicts = {}
    pooled = max(1e-9, statistics.mean([v["std"] for v in stats.values()]))
    for k, v in stats.items():
        if k == control_key: continue
        d = v["mean"] - c["mean"]
        verdicts[k] = "ADOPT" if d > pooled else ("REJECT" if d < 0 else "PARK")
        print(f"  {k:>28}: Δ={d:+.3f} (pooled σ={pooled:.3f}) -> {verdicts[k]}")
    return verdicts

def capitalize(kind, family, content, score, note=""):
    '''Record a decision/skill/tool into campaign memory so later phases reuse it.'''
    mem = MemoryLite(root=MEM_ROOT)
    rec = mem.record_artifact(level="campaign", family=family, kind=kind,
                              content=str(content), score=float(score),
                              metrics={"note": note} if note else None)
    print(f"capitalized [{kind}] {family}: {str(content)[:60]} (score={score})")
    return rec


[MODE] LIVE LLM run  ·  model = gpt-5.4-nano
  Trace-Bench: REAL (Trace-Bench bundle adapter; tasks_root=/home/xav/code/Trace-Bench/notebooks/Trace-Bench/benchmarks/LLM4AD/benchmark_tasks; max_examples=1; inner_steps=1; inner_candidates=1; allowed_inner_trainers=['MinibatchAlgorithm', 'PrioritySearch'])
  PR #73 graph/OTEL: ABSENT (graph/OTEL/Sysmon paths cannot run here)
  Global budget: off: optimizer_llm_calls=0/unlimited, eval_llm_calls=0/unlimited, candidates=0/unlimited, wall_time=0.0s/unlimited, stop_policy=return_best
  Scores below reflect a REAL optimizer run.


## Phase 0 — Gates (no training)
The tests, live model preflight, and real Trace-Bench adapter must be green before any experiment.
**Capitalization:** the gate report itself, so later analysis knows the environment the numbers came
from.

In [2]:
import subprocess, sys
gates = {}
r = subprocess.run([sys.executable, "-m", "pytest",
    "tests/unit_tests/test_recursive_opt.py", "tests/unit_tests/test_recursive_spec.py", "-q"],
    capture_output=True, text=True, cwd="..") if pathlib.Path("../tests").exists() else     subprocess.run([sys.executable, "-m", "pytest",
    "tests/unit_tests/test_recursive_opt.py", "tests/unit_tests/test_recursive_spec.py", "-q"],
    capture_output=True, text=True)
gates["tests"] = r.stdout.strip().splitlines()[-1] if r.stdout else r.stderr[-200:]
gates["adapter"] = TB.real_mode_status()
try:
    from opto.features.recursive_opt.traces import require_pr73
    require_pr73(); gates["pr73"] = "MERGED"
except Exception as e:
    gates["pr73"] = f"NOT MERGED ({type(e).__name__})"
print(json.dumps(gates, indent=1)); save_phase("phase0_gates", gates)

{
 "tests": "62 passed in 3.77s",
 "adapter": "REAL (Trace-Bench bundle adapter; tasks_root=/home/xav/code/Trace-Bench/notebooks/Trace-Bench/benchmarks/LLM4AD/benchmark_tasks; max_examples=1; inner_steps=1; inner_candidates=1; allowed_inner_trainers=['MinibatchAlgorithm', 'PrioritySearch'])",
 "pr73": "NOT MERGED (RuntimeError)"
}



## Phase 1 — Trainer cell *(sets the campaign default under a bounded adapter)*
Specs differ **only** in `fixed.trainer` (trainer is *not* trainable here — paired science),
run under the bounded live smoke budget on `optimization_control`. The measured arms are restricted
to the nested trainers that the current Trace-Bench adapter allowlist can execute cleanly:
`MinibatchAlgorithm` and `PrioritySearch`.

**Lesson learned from probes.** `POLCA` and `ParetobasedPS` are still registered as valid config
labels, but under this smoke budget they are compatibility-only arms: scoring them would measure the
allowlist guard, not trainer performance. Widen `tracebench.allowed_inner_trainers` only when you are
ready for a full, potentially expensive nested benchmark run.

**PR capitalization:** this phase converts a runnable trainer comparison into a reusable campaign
asset: the measured winner becomes `P1_WINNER` (a `kind="decision"` artifact) that later phases read.


In [3]:

TRAINERS = MEASURED_TRAINERS + COMPATIBILITY_ONLY_TRAINERS
register_config_values("trainer", TRAINERS + ["StreamingPrioritySearch"])
def p1_spec(trainer):
    return {"families": FAMILIES, "budget": BUDGET, "scoring": SCORING,
        "prior_promotion": PROMOTION, "memory_root": MEM_ROOT, "reuse_priors": False,
        "tracebench": TRACEBENCH,
        "levels": [{"id": f"o1_{trainer}", "surface": "config", "family": "optimization_control",
                    "targets": ["batch_design","batch_size","memory_policy"],
                    "fixed": {"trainer": trainer, "optimizer": "OptoPrimeV2", "trace_type": "internal"},
                    "iterations": 4}]}
print("measured trainer arms:", MEASURED_TRAINERS)
print("compatibility-only under this adapter:", COMPATIBILITY_ONLY_TRAINERS)
p1 = run_variants(p1_spec, MEASURED_TRAINERS, lambda t: f"o1_{t}")
if p1: save_phase("phase1", p1)


measured trainer arms: ['MinibatchAlgorithm', 'PrioritySearch']
compatibility-only under this adapter: ['POLCA', 'ParetobasedPS']
PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 1 agents on 1 inputs:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  1.98it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  1.98it/s]

...

Evaluating agent:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent: 100%|██████████| 1/1 [00:00<00:00,  1.30it/s]

Evaluating agent: 100%|██████████| 1/1 [00:00<00:00,  1.30it/s]

[Step 0] Test/test_score: -2091.8
[Step 0] Algo/Average train score: -2091.8
[Step 0] Update/n_iters: 0
[Step 0] Update/short_term_memory_size: 0
[Step 0] Update/long_term_memory_size: 1
[Step 0] Update/using_short_term_memory: False
[Step 0] Update/using_long_term_memory: True
[Step 0] Update/total_samples: 0
[Step 0] Update/best_candidate_priority: inf
[Step 0] Update/best_candidate_num_rollouts: 0
[Step 0] Update/num_exploration_candidates: 1
[Step 0] Update/exploration_candidates_mean_priority: inf
[Step 0] Update/exploration_candidates_average_num_rollouts: 0.0
[Step 0] Sample/mean_score: -2091.8
[Step 0] Sample/num_samples: 1
[Step 0] Sample/self.n_epochs: 0
[Step 0] Algo/Number of training samples: 1
[Step 0] Parameter/__code:49: import numpy as np

def priority(item: float, bins: np.ndarray) -> np.ndarray:
    """Returns priority with which we want to add item to each bin.
    Args:
        item: Size of item to be added to the bin.
        bins: Array of capacities for each bi

PrioritySearch = 0.000 ± 0.000 (n=1)


In [4]:

p1 = load_phase("phase1") or {}
if p1:
    print("Phase 1 — measured trainer comparison (normalized improvement over default cfg)")
    for k,v in p1.items(): print(f"  {k:>22}: {v['mean']:+.3f} ± {v['std']:.3f} (n={v['n']})")
    if COMPATIBILITY_ONLY_TRAINERS:
        print("  compatibility-only, not interpreted as performance:", ", ".join(COMPATIBILITY_ONLY_TRAINERS))
    verdicts = decide(p1, "MinibatchAlgorithm")
    P1_WINNER = max(p1, key=lambda k: p1[k]["mean"])
    capitalize("decision", "*", f"trainer={P1_WINNER}", p1[P1_WINNER]["mean"],
               note="Phase-1 measured winner; campaign default trainer")
else:
    P1_WINNER = "PrioritySearch"; print("no Phase-1 data yet -> default", P1_WINNER)


Phase 1 — measured trainer comparison (normalized improvement over default cfg)
      MinibatchAlgorithm: +0.000 ± 0.000 (n=1)
          PrioritySearch: +0.000 ± 0.000 (n=1)
  compatibility-only, not interpreted as performance: POLCA, ParetobasedPS
                PrioritySearch: Δ=+0.000 (pooled σ=0.000) -> PARK
capitalized [decision] *: trainer=MinibatchAlgorithm (score=0.0)


## Phase 2 — Tracing strategies *(validates PR#73; discovers hybridization)*
**Gate:** `require_pr73()`. **If NOT merged → PARK, but capitalize anyway:** (i) the Phase-1
internal-trace numbers *are* the `trace_type=internal` arm — they carry forward as the baseline arm
of this phase, so nothing is wasted; (ii) the paired specs below are validated now and stored, so
the phase is one keystroke once the PR lands; (iii) the campaign proceeds with `trace_type=internal`
as the *known-good* setting (an explicit, recorded assumption — not a silent default).
**If merged:** 2a paired cells (internal/otel/hybrid) → 2b discovery (`targets=["trace_type"]`,
read the per-family choice from lineage) → 2c confirm the discovered mix with one paired cell.

In [5]:
try:
    from opto.features.recursive_opt.traces import require_pr73
    require_pr73(); PR73 = True
except Exception: PR73 = False
TRACES = ["internal", "otel", "hybrid"]
def p2_spec(tt):
    s = p1_spec(P1_WINNER); lvl = s["levels"][0]
    lvl["id"] = f"o1_tt_{tt}"; lvl["fixed"]["trace_type"] = tt; return s
for tt in TRACES: validate_spec(p2_spec(tt))   # specs are ready regardless of PR73
if PR73:
    p2 = run_variants(p2_spec, TRACES, lambda t: f"o1_tt_{t}")
    if p2:
        save_phase("phase2", p2); decide(p2, "internal")
        # 2b discovery: let O1 choose trace_type per family, then O2 across families
        disc = {"families": FAMILIES, "budget": BUDGET, "scoring": SCORING,
                "memory_root": MEM_ROOT, "prior_promotion": PROMOTION, "tracebench": TRACEBENCH,
                "levels": [
                  {"id":"o1_disc","surface":"config","family":"optimization_control",
                   "targets":["trace_type"],"constraints":{"trace_type":TRACES},
                   "fixed":{"trainer":P1_WINNER,"optimizer":"OptoPrimeV2"},"iterations":4},
                  {"id":"o2_mix","surface":"family_policy","family":"*",
                   "targets":["trace_type"],"iterations":2}]}
        validate_spec(disc)
        out = run_spec(disc)
        print("discovered mix:", out["results"]["o2_mix"]["artifact"])
        capitalize("decision","*",out["results"]["o2_mix"]["artifact"],
                   out["results"]["o2_mix"]["score"], note="2b discovery — confirm via 2c before adopting")
else:
    print("PR#73 NOT merged -> Phase 2 PARKED.")
    capitalize("assumption","*","trace_type=internal (PR73 unmerged; P1 numbers = internal arm)",
               (load_phase("phase1") or {}).get(P1_WINNER,{}).get("mean",0.0),
               note="Re-run Phase 2 cells when require_pr73() passes")

PR#73 NOT merged -> Phase 2 PARKED.
capitalized [assumption] *: trace_type=internal (PR73 unmerged; P1 numbers = internal ar (score=0.0)



## Phase 3 — Active search / priors at startup *(transfer)*
Warm vs cold on a **new** family using the priors Phases 1–2 promoted. The promotion **score gate**
(`prior_promotion.min_score`) prevents flat/failed M1 episodes from becoming M3 family priors.

**Lesson learned from probes.** Score-gated promotion is necessary but not sufficient: artifact reuse
can still be neutral or harmful on a flat/new family. This phase now capitalizes memory reuse only when
the paired warm run beats the cold run.


In [6]:

def p3_spec(mode):
    return {"families": FAMILIES, "budget": BUDGET, "scoring": SCORING,
        "prior_promotion": PROMOTION, "tracebench": TRACEBENCH,
        "memory_root": MEM_ROOT if mode=="warm" else "./mem_cold",
        "reuse_priors": mode=="warm",
        "levels": [{"id": f"o1_{mode}", "surface": "config", "family": "reasoning_control",
                    "targets": ["batch_design","batch_size","memory_policy"],
                    "fixed": {"trainer": P1_WINNER, "optimizer": "OptoPrimeV2"}, "iterations": 4}]}
p3 = run_variants(p3_spec, ["cold","warm"], lambda m: f"o1_{m}")
if p3:
    save_phase("phase3", p3); decide(p3, "cold")
    if "warm" in p3 and p3["warm"]["mean"] > p3["cold"]["mean"]:
        capitalize("decision","reasoning_control",f"reuse_priors Δ={p3['warm']['mean']-p3['cold']['mean']:+.3f}",
                   p3["warm"]["mean"], note="value of memory at startup")
    else:
        print("memory reuse not capitalized: warm did not beat cold under this bounded run")


PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 1 agents on 1 inputs:   0%|          | 0/1 [00:00<?, ?it/s]

/home/xav/miniconda3/envs/humanllm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:01<00:00,  1.55s/it]

...

Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:01<00:00,  1.65s/it]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:01<00:00,  1.65s/it]

[Step 0] Average test score: 0.0


warm = 0.020 ± 0.000 (n=1)
                          warm: Δ=+0.013 (pooled σ=0.000) -> ADOPT
capitalized [decision] reasoning_control: reuse_priors Δ=+0.013 (score=0.019999999999999962)


## Phase 4 — AgenticTrace (tool-calling optimizer)
Paired: same spec ± `agentic` (tools wired from memory via `agentic_optimizer_factory`), equal
`optimizer_llm_calls` budget. This **starts the AgenticTrace workstream** with a measurable
baseline. **Capitalization:** tools that helped are saved as `kind="tool"` artifacts — Phase-3
reuse re-arms them automatically in every later run, merged PRs or not.

In [7]:
def p4_spec(mode):
    s = p1_spec(P1_WINNER); lvl = s["levels"][0]
    lvl["id"] = f"o1_{mode}"; s["budget"] = {**BUDGET, "optimizer_llm_calls": 16}
    if mode=="tools": lvl["agentic"] = True; lvl["tools"] = ["trace_search","note"]
    return s
p4 = run_variants(p4_spec, ["plain","tools"], lambda m: f"o1_{m}")
if p4:
    save_phase("phase4", p4); decide(p4, "plain")
    if "tools" in p4 and p4["tools"]["mean"] > p4["plain"]["mean"]:
        capitalize("tool","optimization_control","trace_search", p4["tools"]["mean"],
                   note="tool-evidence improved optimization at equal LLM budget")

PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 1 agents on 1 inputs:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  1.45it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  1.44it/s]

...

Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  2.01it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  2.01it/s]

[Step 0] Average test score: -2091.8


tools = 0.000 ± 0.000 (n=1)
                         tools: Δ=+0.000 (pooled σ=0.000) -> PARK



## Phase 5 — Async trainers *(efficiency, not quality)*
Primary metric: **wall-time to a fixed smoke run**; quality must already be tied in Phase 1.
This cell now times only trainer arms supported by the current nested Trace-Bench adapter. Unsupported
or not-yet-integrated trainers are not included, because timing a budget guard/no-op path would produce
misleading near-zero timings.


In [8]:

from opto.features.recursive_opt.optimize import optimize
from opto.features.recursive_opt.tracebench import make_dataset
def p5_run(num_threads, trainer):
    mem = MemoryLite(root=MEM_ROOT)
    lvl = compile_level({"id":"p5","surface":"config","family":"optimization_control",
        "targets":["batch_design","batch_size"],
        "fixed":{"trainer":trainer,"optimizer":"OptoPrimeV2"}}, mem, FAMILIES)
    t0 = time.time()
    optimize(lvl, make_dataset([FAMILIES["optimization_control"][0]], repeats=4),
             iterations=4, num_threads=num_threads)
    return time.time()-t0
TIMED_TRAINERS = [P1_WINNER]
p5 = {f"{tr}/threads={n}": p5_run(n, tr)
      for tr in TIMED_TRAINERS for n in (1, 8)}
save_phase("phase5", p5)
for k,v in p5.items(): print(f"  {k:>38}: {v:7.1f}s")
if p5: print("Timing is interpreted only for supported trainer paths; quality comes from Phase 1.")


PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 1 agents on 1 inputs (Running sequentially).


Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  1.20it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  1.19it/s]

...

Evaluating agent:  25%|██▌       | 1/4 [00:05<00:15,  5.25s/it]

Evaluating agent:  75%|███████▌  | 3/4 [00:05<00:01,  1.42s/it]

Evaluating agent: 100%|██████████| 4/4 [00:05<00:00,  1.01it/s]

Evaluating agent: 100%|██████████| 4/4 [00:05<00:00,  1.40s/it]

[Step 3] Test/test_score: -2091.8
[Step 3] Algo/Average train score: -2091.8
[Step 3] Update/n_iters: 3
[Step 3] Update/short_term_memory_size: 0
[Step 3] Update/long_term_memory_size: 7
[Step 3] Update/using_short_term_memory: False
[Step 3] Update/using_long_term_memory: True
[Step 3] Update/total_samples: 12
[Step 3] Update/best_candidate_priority: inf
[Step 3] Update/best_candidate_mean_score: -2091.8
[Step 3] Update/best_candidate_num_rollouts: 3
[Step 3] Update/num_exploration_candidates: 1
[Step 3] Update/exploration_candidates_mean_priority: inf
[Step 3] Update/exploration_candidates_mean_score: -2091.8
[Step 3] Update/exploration_candidates_average_num_rollouts: 3.0
[Step 3] Sample/mean_score: -2091.8
[Step 3] Sample/num_samples: 1
[Step 3] Sample/self.n_epochs: 0
[Step 3] Algo/Number of training samples: 4
[Step 3] Parameter/level_config:7: batch_design: random
batch_size: 4
            MinibatchAlgorithm/threads=1:    56.7s
            MinibatchAlgorithm/threads=8:    53.2s



## Phase 6 — skills.md (distill → validate as a prior)
Distilled per family from campaign memory (best artifacts + recurring failures) into a structured
SKILL.md; validated with one paired cell (`starting_artifact` "" vs skill).

**Lesson learned from probes.** A distilled skill is an asset only if paired validation improves the
score. Ties are kept as observations but are not capitalized as reusable skills.


In [9]:
def distill_skill(family):
    mem = MemoryLite(root=MEM_ROOT)
    best = mem.best_artifact(family=family)
    fails = mem.similar_failures(family=family, k=3)
    lines = [f"# SKILL - {family}", "", "## Best known setup"]
    if best: lines += [f"(score={best.score:.3f})", "```", str(best.content), "```"]
    lines += ["", "## Known failure modes"] + [f"- {e.feedback[:140]}" for e in fails]
    lines += ["", "## Procedure", "1. Start from the best known setup above.",
              "2. Verify outputs against the failure modes before accepting a candidate."]
    return "\n".join(lines)
SKILL_FAMILY = "optimization_control"
SKILL = distill_skill(SKILL_FAMILY); print(SKILL[:400])
def p6_spec(mode):
    s = p1_spec(P1_WINNER); lvl = s["levels"][0]; lvl["id"] = f"o1_{mode}"
    if mode=="skill": lvl["fixed"]["starting_artifact"] = SKILL
    return s
p6 = run_variants(p6_spec, ["plain","skill"], lambda m: f"o1_{m}")
if p6:
    save_phase("phase6", p6); decide(p6, "plain")
    if "skill" in p6 and p6["skill"]["mean"] > p6["plain"]["mean"]:
        capitalize("skill",SKILL_FAMILY,SKILL,p6["skill"]["mean"],
                   note="validated SKILL.md (paired vs empty start)")
    else:
        print("skill not capitalized: paired validation did not improve over plain start")


# SKILL - optimization_control

## Best known setup
(score=0.000)
```
batch_design: failure_balanced
batch_size: 16
memory_policy: retrieval
```

## Known failure modes
- [real_trace_bench:llm4ad:online_bin_packing_local] inner_steps=1; cfg applied through Trace trainer. train_dataset: mean over 1 real example
- [real_trace_bench:llm4ad:online_bin_packing_local] inner_steps=1; cfg applied through 
PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 1 agents on 1 inputs:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  1.49it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  1.49it/s]

...

Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  1.04it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00,  1.04it/s]

[Step 0] Average test score: -2091.8


skill = 0.000 ± 0.000 (n=1)
                         skill: Δ=+0.000 (pooled σ=0.000) -> PARK
skill not capitalized: paired validation did not improve over plain start


## Phase 7 — Terminal-Bench 2 onboarding *(parallel track — never blocks 1–6)*
TB2 is **not** a Trace-Bench family yet: the accessible path is (a) an adapter honoring the exact
`register_task_adapter` contract on 2–3 sandboxed terminal tasks with deterministic checks, then
(b) treating `terminal` as a new family in the Phase-3 transfer spec, seeded by the Phase-6 skill
and the promoted O3 prior. **Capitalization:** everything Phases 1–6 banked (trainer decision,
trace assumption/mix, priors, tools, skill) is the warm start — TB2 begins where the campaign is,
not from zero.

In [10]:
class TB2AdapterTemplate:
    '''Contract template: implement run_task (and optionally agent_fn) over a local
    terminal harness; scoring must be deterministic checks (cf. make_code_evaluator).'''
    status = "tb2-template (not implemented)"
    def run_task(self, cfg, task_id):
        raise NotImplementedError("wire a sandboxed terminal harness here")
tb2_spec = {"families": {**FAMILIES, "terminal": ["tb2:hello_fs", "tb2:grep_pipeline"]},
    "budget": BUDGET, "scoring": SCORING, "prior_promotion": PROMOTION,
    "memory_root": MEM_ROOT, "reuse_priors": True,
    "levels": [{"id":"o1_tb2","surface":"config","family":"terminal",
                "targets":["batch_design","batch_size","memory_policy"],
                "fixed":{"trainer":P1_WINNER,"optimizer":"OptoPrimeV2",
                         "starting_artifact": distill_skill("optimization_control")},
                "iterations": 4}]}
validate_spec(tb2_spec); print("TB2 transfer spec validated — runnable the day the adapter exists.")

TB2 transfer spec validated — runnable the day the adapter exists.


## Campaign decision board

In [11]:
board = {}
for ph in ["phase0_gates","phase1","phase2","phase3","phase4","phase5","phase6"]:
    d = load_phase(ph)
    board[ph] = "no data" if d is None else (d if ph=="phase0_gates" else
        {k:(f"{v['mean']:+.3f}±{v['std']:.3f}" if isinstance(v,dict) and "mean" in v else v)
         for k,v in d.items()})
print(json.dumps(board, indent=1))
mem = MemoryLite(root=MEM_ROOT)
print("\ncapitalized assets:", mem.summary())
for kind in ("decision","assumption","tool","skill"):
    for a in mem.artifact_history(kind=kind):
        print(f"  [{kind}] {a.family}: {str(a.content)[:70]} (score={a.score:.3f})")

{
 "phase0_gates": {
  "tests": "62 passed in 3.77s",
  "adapter": "REAL (Trace-Bench bundle adapter; tasks_root=/home/xav/code/Trace-Bench/notebooks/Trace-Bench/benchmarks/LLM4AD/benchmark_tasks; max_examples=1; inner_steps=1; inner_candidates=1; allowed_inner_trainers=['MinibatchAlgorithm', 'PrioritySearch'])",
  "pr73": "NOT MERGED (RuntimeError)"
 },
 "phase1": {
  "MinibatchAlgorithm": "+0.000\u00b10.000",
  "PrioritySearch": "+0.000\u00b10.000"
 },
 "phase2": "no data",
 "phase3": {
  "cold": "+0.007\u00b10.000",
  "warm": "+0.020\u00b10.000"
 },
 "phase4": {
  "plain": "+0.000\u00b10.000",
  "tools": "+0.000\u00b10.000"
 },
 "phase5": {
  "MinibatchAlgorithm/threads=1": 56.748968839645386,
  "MinibatchAlgorithm/threads=8": 53.17313504219055
 },
 "phase6": {
  "plain": "+0.000\u00b10.000",
  "skill": "+0.000\u00b10.000"
 }
}

capitalized assets: {'episodes': 1076, 'artifacts': 53, 'families': ['llm4ad:online_bin_packing_local', 'optimization_control', 'reasoning_control'], 'prior